In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import string

nltk.download("punkt")

# Load QA dataset
df = pd.read_excel("../data/noticiasQA2.xlsx")

# Load DeepSeek (or any generative LLM)
model_name = "deepseek-ai/deepseek-llm-7b-base"  # You can use mistralai/Mistral-7B-Instruct-v0.2, etc.

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Normalize function for EM and F1
def normalize(text):
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return word_tokenize(text)

def f1_score(prediction, ground_truth):
    pred_tokens = normalize(prediction)
    truth_tokens = normalize(ground_truth)
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    return 2 * (precision * recall) / (precision + recall)

def exact_match(prediction, ground_truth):
    return int(normalize(prediction) == normalize(ground_truth))

# Evaluation
em_total = 0
f1_total = 0
n = len(df)

# Inference loop
for _, row in tqdm(df.iterrows(), total=n):
    context = row["context"]
    question = row["question"]
    ground_truth = str(row["answer"])

    # Build Spanish QA prompt
    prompt = f"""Contesta la siguiente pregunta usando el contexto provisto.

Contexto: {context}

Pregunta: {question}

Respuesta:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        temperature=0.7,
        top_p=0.95
    )
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_answer = output_text.split("Respuesta:")[-1].strip()

    print("RESULTS")
    print(question)
    print(generated_answer)

    em_total += exact_match(generated_answer, ground_truth)
    f1_total += f1_score(generated_answer, ground_truth)

# Results
print(f"\nExact Match: {em_total / n * 100:.2f}%")
print(f"F1 Score: {f1_total / n * 100:.2f}%")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\guill\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]C:\Users\guill\Documents\GitHub\nlp-media-framing\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\guill\Documents\GitHub\nlp-media-framing\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
  5%|▍         | 1/21 [02:11<43:56, 131.83s/it]Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


RESULTS
¿Quién intervino en el segundo día de la Cumbre CELAC-UE?
Gustavo Petro


 10%|▉         | 2/21 [04:23<41:44, 131.82s/it]Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


...

RESULTS
¿Qué tipo de comercio se ha presentado en el lado europeo?
El presidente Petro señaló que el comercio de productos que no tengan detrás deforestación ha sido presentado en el lado europeo. Esto significa que los productos comercializados en Europa no deben tener como base la deforestación.



En este sentido, el presidente Petro señaló que la agricultura ha deforestado tanto en Europa como en América Latina. Sin embargo, el com


 95%|█████████▌| 20/21 [43:44<02:15, 135.20s/it]Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


RESULTS
¿Qué ha hecho toda agricultura según el mandatario colombiano?



100%|██████████| 21/21 [46:01<00:00, 131.48s/it]

RESULTS
¿Cuál debería ser el criterio de comercio según el mandatario colombiano?


Exact Match: 0.00%
F1 Score: 21.44%
